**Cell 1**

# 03 — Dataset Health Check

The unified, pre-split dataset was built by notebook 02 at `data/dataset/`. This notebook **inspects** that dataset before training: class balance per split, bounding-box size distribution, image dimensions, and split-leakage checks.

No files are modified here — it is diagnostic only. If any of the checks fail, go back to notebook 02 and re-run after fixing the source data.

In [ ]:
# Cell 2 — install dependencies for this notebook
%pip install -q pandas pillow matplotlib seaborn

In [ ]:
# Cell 3 — setup
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
DST = Path('../data/dataset')
SPLITS = ['train', 'val', 'test']
assert (DST / 'data.yaml').exists(), 'run notebook 02 first to build data/dataset/'

**Cell 4**

## 3.1 Load every label into one DataFrame

In [ ]:
# Cell 5 — aggregate labels + image dims across all splits
rows = []
for split in SPLITS:
    img_dir = DST / 'images' / split; lbl_dir = DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        with Image.open(img) as im:
            W, H = im.size
        lbl = lbl_dir / (img.stem + '.txt')
        boxes = 0
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cid, xc, yc, w, h = map(float, parts)
                rows.append({'split': split, 'file': img.name, 'W': W, 'H': H,
                             'class': CLASSES[int(cid)], 'bx_w': w, 'bx_h': h, 'bx_area': w*h})
                boxes += 1
        if boxes == 0:
            rows.append({'split': split, 'file': img.name, 'W': W, 'H': H,
                         'class': None, 'bx_w': None, 'bx_h': None, 'bx_area': None})

df = pd.DataFrame(rows)
print('total rows (one per box, or one per empty image):', len(df))
df.head()

**Cell 6**

## 3.2 Per-split class balance (table + chart)

In [ ]:
# Cell 7 — image counts per class per split + a bar chart
per_img = df.dropna(subset=['class']).drop_duplicates(subset=['split', 'file', 'class'])
table = per_img.groupby(['split', 'class']).size().unstack(fill_value=0).reindex(SPLITS)
display(table)

ax = table.plot.bar(figsize=(8, 4))
ax.set_ylabel('images containing class'); ax.set_title('Per-class image count across splits')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Cell 8**

## 3.3 Box size distribution per class
Very small boxes (normalized area < 0.001 → roughly < 20×20 px on a 640 image) are hard for the detector and worth flagging.

In [ ]:
# Cell 9 — box area histogram per class
boxes = df.dropna(subset=['bx_area'])
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=boxes, x='bx_area', hue='class', bins=40, stat='density', common_norm=False, ax=ax)
ax.set_title('Normalized box area, per class'); ax.set_xlabel('box area (w*h, normalized)')
plt.tight_layout(); plt.show()

tiny = boxes[boxes['bx_area'] < 1e-3]
print(f'tiny boxes (<0.001 area): {len(tiny)} / {len(boxes)}')

**Cell 10**

## 3.4 Image dimension distribution
Helps confirm the chosen training `imgsz=640` is reasonable for the source resolutions.

In [ ]:
# Cell 11 — W/H scatter per split
dims = df.drop_duplicates(subset=['split', 'file'])[['split', 'W', 'H']]
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(data=dims, x='W', y='H', hue='split', alpha=0.5, ax=ax)
ax.set_title('Source image dimensions'); ax.set_xlabel('width'); ax.set_ylabel('height')
plt.tight_layout(); plt.show()
dims.describe()

**Cell 12**

## 3.5 Empty / degenerate labels
YOLO accepts images with no labels (they count as background), but too many of them bias the detector. Confirm the split isn't accidentally unlabeled.

In [ ]:
# Cell 13 — empty labels + near-degenerate boxes per split
by_split = df.groupby('split')
summary = pd.DataFrame({
    'images':        by_split['file'].nunique(),
    'empty_labels':  by_split.apply(lambda g: g[g['class'].isna()]['file'].nunique()),
    'boxes':         by_split['bx_area'].count(),
    'tiny_boxes':    by_split.apply(lambda g: int((g['bx_area'] < 1e-3).sum())),
}).reindex(SPLITS)
summary

**Cell 14**

## 3.6 Cross-split leakage
Same filename must not appear in more than one split (simple leakage guard).

In [ ]:
# Cell 15 — detect filenames present in multiple splits
files = df.drop_duplicates(subset=['split', 'file'])[['split', 'file']]
dups = files.groupby('file')['split'].nunique()
leaked = dups[dups > 1]
print('filenames appearing in multiple splits:', len(leaked))
leaked.head()

**Cell 16**

### Health checklist
- [ ] Every class present in every split
- [ ] No split dominated by one class (skew < ~2×)
- [ ] `empty_labels` is 0 (or intentionally small)
- [ ] `tiny_boxes` / `boxes` ≤ 5%
- [ ] `leaked` is 0
- [ ] Image dimensions consistent with `imgsz=640` training

If all checks pass, proceed to **notebook 04** for training.